|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>One engine<h1>|
|<h2>Lecture:</h2>|<h1><b>Code challenge: build the scheduler<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

Build the scheduler of the capstone engine, as a token counter.

No model and no GPU. The scheduler decides every step, and it is the part of
stage 22 that is hardest to test on a real model. Here you can test it to
destruction in a second.

In [ ]:
from collections import deque
from dataclasses import dataclass

@dataclass(eq=False)
class Sequence:
    seq_id: int
    prompt_len: int
    max_tokens: int
    prefix_len: int = 0      # tokens of a shared system prompt
    generated: int = 0       # output tokens so far
    computed: int = 0        # tokens whose K and V are in the cache
    blocks: int = 0

    @property
    def pending(self):
        """The tokens that the model must still process."""
        return self.prompt_len + self.generated - self.computed

BLOCK = 16

def blocks_for(num_tokens):
    return -(-num_tokens // BLOCK)                # ceiling division

def new_blocks(seq, num_tokens):
    """The blocks that seq needs, in addition to its blocks, for num_tokens more."""
    return blocks_for(seq.computed + num_tokens) - seq.blocks

class StepPlan:
    """The plan of one step: the sequences that run, and the tokens of each."""
    def __init__(self, budget, free):
        self.items = []              # (seq, num_tokens)
        self.budget_left = budget
        self.free = free             # free blocks after this plan
        self.preempted = 0

    def fits(self, seq, num_tokens):
        return new_blocks(seq, num_tokens) <= self.free

    def add(self, seq, num_tokens):
        needed = new_blocks(seq, num_tokens)
        self.free -= needed
        seq.blocks += needed
        self.items.append((seq, num_tokens))
        self.budget_left -= num_tokens

# Exercise 1: the one rule

Running sequences first. Each one gets min(pending, budget left) tokens. Then
admit waiting sequences in order, while there is budget, a free slot and
enough blocks.

In [ ]:
def schedule_running(plan, running):
    """Each running sequence gets min(pending, budget left) tokens, if the
    blocks for them exist."""
    for seq in running:
        if plan.budget_left == 0:
            break
        num_tokens = 
        if plan.fits(seq, num_tokens):
            plan.add(seq, num_tokens)

def admit_waiting(plan, running, waiting, max_seqs):
    """Admit waiting sequences in order, while there is budget, a free slot
    and enough blocks."""
    while waiting and plan.budget_left > 0 and len(running) < max_seqs:
        


def plan_step(running, waiting, budget, free, max_seqs=64):
    """The one rule, with no preemption. -> a StepPlan."""
    plan = StepPlan(budget, free)
    schedule_running(plan, running)
    admit_waiting(plan, running, waiting, max_seqs)
    return plan

def finish_step(plan, running):
    """After the model runs: count the tokens, sample where nothing is
    pending, and free the blocks of finished sequences. -> the free blocks."""
    free = plan.free
    for seq, num_tokens in plan.items:
        seq.computed += num_tokens
        if seq.pending == 0:
            seq.generated += 1
    for seq in [seq for seq in running if seq.generated >= seq.max_tokens]:
        running.remove(seq)
        free += seq.blocks
        seq.blocks = 0
    return free

# A check by hand: one decode, and one new prompt of 40 tokens, budget 16.
decoding = Sequence(0, prompt_len=5, max_tokens=10, generated=2, computed=6, blocks=1)
new_prompt = Sequence(1, prompt_len=40, max_tokens=4)
plan = plan_step([decoding], deque([new_prompt]), budget=16, free=100)
scheduled = [(seq.seq_id, num_tokens) for seq, num_tokens in plan.items]
print(scheduled, 'free', plan.free)
assert scheduled == [(0, 1), (1, 15)]

# Exercise 2: preemption by recompute

Make the pool too small. When the blocks run out, preempt the NEWEST running
sequence, and put it at the front of the queue with everything pending.

In [ ]:
def preempt_newest(plan, running, waiting):
    """Preempt the NEWEST running sequence. Give back its blocks, set its
    computed tokens to 0, and put it at the FRONT of the queue. -> the victim."""
    


def schedule_running_with_preemption(plan, running, waiting):
    """As schedule_running. When a sequence cannot get its blocks, preempt
    until the blocks exist, or until the sequence preempts itself."""
    index = 0
    while index < len(running) and plan.budget_left > 0:
        seq = running[index]
        num_tokens = min(seq.pending, plan.budget_left)
        while not plan.fits(seq, num_tokens):
            if preempt_newest(plan, running, waiting) is seq:
                break
        if seq not in running:
            continue
        plan.add(seq, num_tokens)
        index += 1

def plan_step_preempt(running, waiting, budget, free, max_seqs=64):
    """The one rule, with preemption by recompute. -> a StepPlan."""
    plan = StepPlan(budget, free)
    schedule_running_with_preemption(plan, running, waiting)
    # Admit as in Exercise 1, but not after a preemption.
    
    return plan

def run(pool_blocks, budget, seqs, max_steps=100000):
    """Run all sequences to the end. -> the counts of the run."""
    waiting, running, free = deque(seqs), [], pool_blocks
    stats = dict(steps=0, work=0, preemptions=0, largest_step=0)
    while (waiting or running) and stats['steps'] < max_steps:
        plan = plan_step_preempt(running, waiting, budget, free)
        step_tokens = sum(num_tokens for _, num_tokens in plan.items)
        stats['work'] += step_tokens
        stats['largest_step'] = max(stats['largest_step'], step_tokens)
        stats['preemptions'] += plan.preempted
        free = finish_step(plan, running)
        stats['steps'] += 1
    return stats

def load(num_seqs=60, seed=1):
    rng = np.random.default_rng(seed)
    return [Sequence(seq_id, int(rng.integers(32, 512)), int(rng.integers(16, 256)))
            for seq_id in range(num_seqs)]

print(f'{"pool":>6} {"steps":>6} {"preemptions":>12} {"work / useful":>14}')
for pool_blocks in (5000, 800, 400, 250):
    seqs = load()
    useful_work = sum(seq.prompt_len + seq.max_tokens for seq in seqs)
    stats = run(pool_blocks, 512, seqs)
    assert all(seq.generated >= seq.max_tokens for seq in seqs), 'a request did not finish'
    print(f'{pool_blocks:>6} {stats["steps"]:>6} {stats["preemptions"]:>12} '
          f'{stats["work"] / useful_work:>14.2f}')

# Exercise 3: the prefix cache at admission

Sixty requests share a 512-token system prompt. Count the prefill tokens with
and without a cache that already holds that prompt.

In [ ]:
def admit_with_prefix(seqs, cached_prefix_tokens):
    """Mark the shared prefix as computed at admission.

    Only WHOLE blocks can come from the cache, because the cache hashes whole
    blocks. And leave at least one prompt token to compute: the last token
    must make the logits for the first output token."""
    for seq in seqs:
        usable = 
        cache_hit = 
        seq.computed = cache_hit
        seq.blocks = blocks_for(cache_hit)

SYSTEM_PROMPT = 512

def with_system_prompt(num_seqs=60, seed=2):
    rng = np.random.default_rng(seed)
    return [Sequence(seq_id, SYSTEM_PROMPT + int(rng.integers(8, 64)), 32,
                     prefix_len=SYSTEM_PROMPT)
            for seq_id in range(num_seqs)]

prefill_tokens = []
for cached_tokens in (0, SYSTEM_PROMPT):
    seqs = with_system_prompt()           # the same requests each time
    admit_with_prefix(seqs, cached_tokens)
    prefill_tokens.append(sum(seq.prompt_len - seq.computed for seq in seqs))
without_cache, with_cache = prefill_tokens
print(f'prefill tokens without the cache: {without_cache:,}')
print(f'prefill tokens with the cache:    {with_cache:,}  '
      f'({100 * with_cache / without_cache:.0f}% of the work)')

# Exercise 4: the budget is a dial

Sweep the token budget. Count the steps, and the largest step in tokens. A
large step is a long pause for every sequence that decodes in it.

In [ ]:
print(f'{"budget":>7} {"steps":>6} {"largest step (tokens)":>22}')
for budget in (64, 128, 256, 512, 1024, 2048):
    stats = run(5000, budget, load())
    print(f'{budget:>7} {stats["steps"]:>6} {stats["largest_step"]:>22}')

### Before you open the solution

1. In Exercise 2, work / useful work climbs as the pool shrinks. Where does
   the extra work come from, token by token?
2. The admission loop admits nothing after a preemption. What happens if you
   remove that condition? Try it, and count the preemptions.
3. In Exercise 3, why must at least one prompt token stay pending, even when
   the whole prompt is in the cache?
4. In Exercise 4, which budget would you choose, and what does your answer
   depend on? Name the two quantities that trade against each other.